# Best Single Depth-Limited CIFAR-10 Model

This streamlined notebook keeps only the selected best single model, `UltraWideScaledTail10`. The previous multi-model and ensemble code has been removed so the notebook focuses on one architecture, its layer-budget audit, and the reproducibility training entry point.

The companion script [`train_ultrawidescaledtail10_repro.py`](train_ultrawidescaledtail10_repro.py) runs the 8-seed reproducibility sweep for 750 epochs per seed on the 8x A40 RunPod setup.


In [ ]:
from __future__ import annotations

import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from train_ultrawidescaledtail10_repro import (
    MODEL_KEY,
    MODEL_NAME,
    TrainConfig,
    UltraWideScaledTail10,
    build_config,
    count_param_layers,
    n_params,
    run_seed_sweep,
    weighted_layer_rows,
)

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Selected model: {MODEL_NAME} ({MODEL_KEY})")


## Selected Model

`UltraWideScaledTail10` was the strongest single architecture in the prior validation evidence. It uses a wide convolutional trunk, then spends the remaining weighted layers in a low-resolution residual tail.

Only convolutional and linear modules count toward the project depth budget. Normalization, pooling, activation, gating, residual additions, stochastic depth, and scalar residual parameters do not count.


In [ ]:
model = UltraWideScaledTail10()
model.eval()

with torch.no_grad():
    dummy_out = model(torch.zeros(2, 3, 32, 32))

model_audit = pd.DataFrame(
    [
        {
            "model": MODEL_NAME,
            "weighted_layers": count_param_layers(model),
            "params_m": n_params(model) / 1_000_000,
            "dummy_output_shape": tuple(dummy_out.shape),
            "within_budget": count_param_layers(model) <= 10,
        }
    ]
)

display(model_audit.style.format({"params_m": "{:.2f}"}))
assert count_param_layers(model) == 10
assert tuple(dummy_out.shape) == (2, 10)


In [ ]:
layer_table = pd.DataFrame(weighted_layer_rows(model))
display(layer_table)


## Embedded Selection Evidence

The retained evidence below is limited to the selected model's prior 500-epoch seed runs. The 750-epoch, 8-seed sweep in the training script is the reproducibility run to refresh this table with the final configuration.


In [ ]:
SELECTED_MODEL_SEED_RESULTS = [
    {
        "seed": 42,
        "selected_oos_val": 96.94,
        "best_val_raw": 96.88,
        "best_val_ema": 96.94,
        "selected_epoch": 459,
        "params_m": 21.337656,
        "layers": 10,
        "source_run": "bw500_ultrawidescaledtail10_s42",
    },
    {
        "seed": 43,
        "selected_oos_val": 97.06,
        "best_val_raw": 97.06,
        "best_val_ema": 97.02,
        "selected_epoch": 481,
        "params_m": 21.337656,
        "layers": 10,
        "source_run": "bw500_ultrawidescaledtail10_s43",
    },
    {
        "seed": 44,
        "selected_oos_val": 96.90,
        "best_val_raw": 96.90,
        "best_val_ema": 96.90,
        "selected_epoch": 486,
        "params_m": 21.337656,
        "layers": 10,
        "source_run": "bw500_ultrawidescaledtail10_s44",
    },
]

seed_df = pd.DataFrame(SELECTED_MODEL_SEED_RESULTS)
summary_df = pd.DataFrame(
    [
        {
            "model": MODEL_NAME,
            "seeds": len(seed_df),
            "mean_oos_val": seed_df["selected_oos_val"].mean(),
            "std_oos_val": seed_df["selected_oos_val"].std(ddof=1),
            "max_oos_val": seed_df["selected_oos_val"].max(),
            "params_m": seed_df["params_m"].iloc[0],
            "layers": seed_df["layers"].iloc[0],
        }
    ]
)

display(
    seed_df.style.format(
        {
            "selected_oos_val": "{:.2f}",
            "best_val_raw": "{:.2f}",
            "best_val_ema": "{:.2f}",
            "params_m": "{:.2f}",
        }
    )
)
display(
    summary_df.style.format(
        {
            "mean_oos_val": "{:.2f}",
            "std_oos_val": "{:.2f}",
            "max_oos_val": "{:.2f}",
            "params_m": "{:.2f}",
        }
    )
)


## 10-Seed, 750-Epoch Reproducibility Run

The reproducibility sweep uses the top-of-file `RUN_*` configuration in `train_ultrawidescaledtail10_repro.py`. The final setting uses seeds `42` through `49`, trains `UltraWideScaledTail10` for 750 epochs per seed, and writes per-seed checkpoints plus `summary.csv` and `summary.json` into `repro_runs_ultrawidescaledtail10_8seed/`.

The only terminal command needed is `python train_ultrawidescaledtail10_repro.py`. If the professor wants a faster verification run, temporarily set `RUN_SEEDS = (42, 43, 44)` near the top of the script, then use the same command.


In [ ]:
REPRO_CONFIG = build_config()

config_df = pd.DataFrame([asdict(REPRO_CONFIG)]).T.rename(columns={0: "value"})
display(config_df)

print("Terminal command:")
print("python train_ultrawidescaledtail10_repro.py")


In [ ]:
RUN_FULL_REPRODUCIBILITY_TRAINING = False

if RUN_FULL_REPRODUCIBILITY_TRAINING:
    sweep_results, sweep_summary = run_seed_sweep(REPRO_CONFIG)
    display(pd.DataFrame(sweep_results)[["seed", "selected_checkpoint", "selected_val_acc", "selected_epoch", "best_val_raw", "best_val_ema", "wall_sec"]])
    display(pd.DataFrame([sweep_summary]))
else:
    print("Full training is disabled in the notebook. Use the terminal command above, or set RUN_FULL_REPRODUCIBILITY_TRAINING=True.")


## Optional Smoke Test

Use this quick check only to verify the training path, data loading, and checkpoint writing. It is not a performance run.


In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    smoke_config = TrainConfig(
        seeds=(42,),
        epochs=2,
        batch_size=128,
        num_workers=0,
        max_train_batches=4,
        use_amp=False,
        output_dir="./smoke_ultrawidescaledtail10",
    )
    smoke_results, smoke_summary = run_seed_sweep(smoke_config)
    display(pd.DataFrame(smoke_results)[["seed", "selected_val_acc", "selected_epoch", "wall_sec"]])
else:
    print("Smoke test is disabled.")


## Final Claim

The final single-model submission is `UltraWideScaledTail10`: a 10-weighted-layer CIFAR-10 model with about 21.34M parameters. The updated reproducibility target is the 8-seed, 750-epoch sweep defined in `train_ultrawidescaledtail10_repro.py`.
